In [2]:
from torchvision import transforms, datasets

preprocessing = transforms.Compose([
    transforms.ToTensor()
])

train = datasets.MNIST(root= './data', train= True, download= True, transform= preprocessing)
test = datasets.MNIST(root='./data', train= False, download= True, transform= preprocessing)

In [3]:
print(len(train))
print(len(test))

60000
10000


In [7]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train, batch_size= 64, shuffle= True, num_workers= 6)
test_loader = DataLoader(test, batch_size= 64, num_workers= 6)

In [11]:
images, labels = next(iter(train_loader))

print(images.size())
print(labels.size())

torch.Size([64, 1, 28, 28])
torch.Size([64])


In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CustomCNN(nn.Module):
    def __init__(self, num_classes= 10):
        super(CustomCNN, self).__init__()
        # First Convolutional Layer
        # in -> 1 * 28 * 28
        self.conv1 = nn.Conv2d(in_channels= 1, out_channels= 32, kernel_size= 3, padding= 1)

        # Second Convolutional Layer
        # in -> 32 * 14 * 14
        self.conv2 = nn.Conv2d(in_channels= 32, out_channels= 64, kernel_size= 3, padding= 1)

        # Pooling
        self.pooling = nn.MaxPool2d(2, 2)

        # Dropout
        self.dropout = nn.Dropout(0.5)

        # First Fully Connected Layer
        # in -> 64 * 7 * 7
        self.fc3 = nn.Linear(64 * 7 * 7, 128)

        # Second Fully Connected Layer (output Layer)
        # in -> 128
        self.fc4 = nn.Linear(128, num_classes)

    def forward(self, x):
        # First Convolutiona Layer
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pooling(x)

        # Second Convolutional Layer
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pooling(x)

        # Flattening
        x = torch.flatten(x, start_dim= 1)

        # First Fully Connected Layer
        x = self.fc3(x)
        x = F.relu(x)
        x = self.dropout(x)

        # Second Fully Connected Layer (output layer)
        x = self.fc4(x)

        return x

In [25]:
# Setting Cuda

device = ('cuda' if torch.cuda.is_available() else 'cpu')

# opject from class (model) and set it to the cuda not the cpu
model = CustomCNN(num_classes= 10).to(device)

# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr= 0.001, weight_decay= 0.0001)        # weight_decay Param is the R2 norm (Ridge Regularization)

In [30]:
# Training Loop

epochs = 8


for epoch in range(epochs):
    model.train()
    
    training_loss = 0
    total = 0
    correct = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        
        training_loss += loss.item() * labels.size(0)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_epoch_loss = training_loss / total
    avg_epoch_acc = correct / total
    
    print(f"Epoch {epoch+1} | loss = {avg_epoch_loss : .4f}, acc = {avg_epoch_acc : .4f}")

Epoch 1 | loss =  0.2872, acc =  0.9121
Epoch 2 | loss =  0.0983, acc =  0.9706
Epoch 3 | loss =  0.0788, acc =  0.9775
Epoch 4 | loss =  0.0624, acc =  0.9815
Epoch 5 | loss =  0.0548, acc =  0.9839
Epoch 6 | loss =  0.0497, acc =  0.9851
Epoch 7 | loss =  0.0443, acc =  0.9867
Epoch 8 | loss =  0.0398, acc =  0.9879


In [34]:
# Testing loop

model.eval()

testing_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        _, predicted = torch.max(outputs, 1)
        
        testing_loss += loss.item() * labels.size(0)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    test_loss = testing_loss / total
    test_acc = correct / total

print(f"Loss = {test_loss : .4f}, acc = {test_acc:.4f}")    

Loss =  0.0257, acc = 0.9923
